import the libraries important for data cleaning and preparation

In [1]:
import pandas as pd
import camelot
import warnings

some tweaks to supress irrelevant warnings and optimize some settings

In [2]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns',  None)

Convert the pdf into csv

In [3]:
tables = camelot.read_pdf('../data/auction2.pdf', pages='all', flavor='lattice')
combined = pd.concat([table.df for table in tables], ignore_index=True)
combined.to_csv('../data/clean_auction2.csv')

import the csv file for manipulation and cleaning give the dataset the correct columns name

In [4]:
df = pd.read_csv('../data/clean_auction2.csv')
df.columns = ['one', 'no', 'rank', 'winners', 'price_per_sqm', 'down_payment_pct', 'subcity', 'district', 'sqm', 'code', 'comment']
print(f"The shape of the first dataset is {df.shape}")

The shape of the first dataset is (664, 11)


drop irrelevant columns

In [5]:
df.drop(columns=['one', 'no', 'rank', 'winners', 'code', 'comment'], inplace=True)

drop irrelevant and empty rows, then rearrange the index

In [6]:
df = df.dropna(subset=['price_per_sqm'])
df = df[df['price_per_sqm'].astype(str).str.contains(r'\d', regex=True, na=False)]
df = df.reset_index(drop=True)

rename the subcity names into the appropriate kinda names

In [7]:
df.loc[0:53, 'subcity'] = 'Kolfe Keranyo'
df.loc[54:68, 'subcity'] = 'Gulele'
df.loc[69:71, 'subcity'] = 'Lideta'
df.loc[72:86, 'subcity'] = 'Yeka'
df.loc[87:95, 'subcity'] = 'Arada'
df.loc[96:269, 'subcity'] = 'Akaki Kality'
df.loc[270:314, 'subcity'] = 'Addis Ketema'
df.loc[315:, 'subcity'] = 'Nefas - Silk Lafto'

inspect the data types and change them to the appropriate ones

In [8]:
df['price_per_sqm'] = df['price_per_sqm'].astype(str).str.replace(',', '').str.strip().astype('float64')
df['down_payment_pct'] = df['down_payment_pct'].astype(str).str.replace('%', '').str.strip().astype('float64')
df['district'] = df['district'].astype(str).str.replace('ዏ', '0')
df['sqm'] = df['sqm'].astype(str).str.replace('ዏ', '0').astype('float64')
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   price_per_sqm     645 non-null    float64
 1   down_payment_pct  645 non-null    float64
 2   subcity           645 non-null    str    
 3   district          645 non-null    str    
 4   sqm               645 non-null    float64
dtypes: float64(3), str(2)
memory usage: 25.3 KB


None

In [10]:
df.to_csv('../data/clean_auction2.csv', index=False)